# NB-EDA-001 · Feature Distribution Analysis
## Cinematic Emotion Lab

**Notebook ID:** NB-EDA-001  
**Type:** Exploratory Data Analysis  
**Date:** 2026-08-01  
**Status:** Public — analysis of public corpus subset only  
**Author:** Bernard G. · Cinematic Emotion Lab  

---

> This notebook analyzes the distribution of acoustic features across the public research subset
> of the Cinematic Emotion Lab corpus. It is a structural analysis — examining how feature values
> are distributed across the corpus and across annotated structural metadata. It does not expose
> per-clip emotion labels or classification results, which are embargoed pending peer-reviewed
> publication.

### What This Notebook Does

- Loads the public acoustic feature subset from `datasets/processed/public_subset_features.csv`
- Characterizes the marginal distribution of each feature group (MFCCs, Chroma, Spectral, RMS, ZCR, Tempo, H/P ratio)
- Analyzes feature variance and inter-feature correlation within the 71-dimensional space
- Produces a corpus-level statistical summary: range, spread, skew, kurtosis per feature
- Visualizes feature distributions by `harmonic_mode` and `narrative_position` (structural metadata — not emotion labels)
- Identifies high-variance and low-variance feature dimensions as a precursor to dimensionality analysis

### What This Notebook Does Not Do

- It does not load or analyze emotion labels (`primary_emotion`, `emotional_intensity` are absent from the public subset)
- It does not perform classification, clustering, or predictive modeling
- It does not expose any results from EXP-002, EXP-003, or EXP-004
- It does not produce figures that characterize emotion-category separability

### Related Notebooks

| Notebook | Description | Status |
|----------|-------------|--------|
| NB-EXP-001 | Feature extraction pipeline | Released v0.1 |
| NB-EDA-001 | This notebook — feature distribution analysis | Released v0.2 |
| NB-EDA-002 | Annotation distribution analysis (full corpus) | Target v0.25 |
| NB-VIZ-001 | Spectrogram intelligence gallery | Target v0.15 |
| NB-VIZ-002 | Chroma and harmonic profile visualization | Target v0.15 |
| NB-VIZ-003 | Feature space UMAP projection | Target v0.3 |
| NB-EXP-002 | Baseline emotion classification | Target v0.3 |


---

## 0. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Cinematic Emotion Lab visual identity
CEL_PALETTE = {
    'background': '#0a0a0a',
    'surface':    '#111111',
    'gold':       '#c9a227',
    'red':        '#8b1a1a',
    'blue':       '#1a3a5c',
    'text':       '#e8e8e8',
    'muted':      '#555555',
    'accent_2':   '#2d5a3d',
    'accent_3':   '#5c3d1a',
}

EMOTION_COLORS = {
    'tension':   '#8b1a1a',
    'triumph':   '#c9a227',
    'grief':     '#1a3a5c',
    'suspense':  '#3d1a5c',
    'joy':       '#2d5a3d',
    'dread':     '#2a1a0a',
    'longing':   '#1a2a4a',
    'ambiguity': '#3a3a3a',
}

NARRATIVE_COLORS = {
    'opening':    '#1a3a5c',
    'rising':     '#2d5a3d',
    'climax':     '#8b1a1a',
    'falling':    '#5c3d1a',
    'resolution': '#3a3a3a',
}

MODE_COLORS = {
    'major':  '#c9a227',
    'minor':  '#1a3a5c',
    'modal':  '#3d1a5c',
    'atonal': '#8b1a1a',
}

# Matplotlib cinematic theme
plt.rcParams.update({
    'figure.facecolor':  CEL_PALETTE['background'],
    'axes.facecolor':    CEL_PALETTE['surface'],
    'axes.edgecolor':    CEL_PALETTE['muted'],
    'axes.labelcolor':   CEL_PALETTE['text'],
    'xtick.color':       CEL_PALETTE['muted'],
    'ytick.color':       CEL_PALETTE['muted'],
    'text.color':        CEL_PALETTE['text'],
    'grid.color':        '#1a1a1a',
    'grid.linewidth':    0.5,
    'font.family':       'monospace',
    'axes.titlesize':    12,
    'axes.labelsize':    10,
})

print('Cinematic Emotion Lab · NB-EDA-001 · Environment initialized')
print(f'NumPy {np.__version__} · Pandas {pd.__version__}')

---

## 1. Load Public Research Subset

The public subset contains 71-dimensional acoustic feature vectors and structural metadata
(harmonic mode, narrative position, tempo category, narrative function). Emotion labels are
absent from this file — they are withheld pending publication.

In [ ]:
import os

SUBSET_PATH = '../../datasets/processed/public_subset_features.csv'

# Feature group column ranges — consistent with NB-EXP-001 extraction schema
FEATURE_GROUPS = {
    'MFCC':       list(range(0, 26)),    # 26 dimensions: 13 coefficients × mean + std
    'Chroma_CQT': list(range(26, 53)),   # 27 dimensions: 12 pitch classes × mean + std + 3 summary stats
    'Spectral':   list(range(53, 59)),   # 6 dimensions: centroid, bandwidth, rolloff × mean + std
    'RMS':        list(range(59, 63)),   # 4 dimensions: mean, std, max, dynamic_range
    'ZCR':        list(range(63, 65)),   # 2 dimensions: mean, std
    'Tempo':      [65],                  # 1 dimension
    'HP_Ratio':   list(range(66, 69)),   # 3 dimensions: mean, std, max
}

METADATA_COLS = ['clip_id', 'harmonic_mode', 'narrative_position',
                 'tempo_category', 'narrative_function']

if os.path.exists(SUBSET_PATH):
    df = pd.read_csv(SUBSET_PATH)
    feature_cols = [c for c in df.columns if c not in METADATA_COLS]
    X = df[feature_cols].values
    meta = df[METADATA_COLS]
    print(f'Public subset loaded: {X.shape[0]} cues × {X.shape[1]} feature dimensions')
    print(f'Structural metadata columns: {list(meta.columns)}')
    print(f'Harmonic mode distribution:\n{meta["harmonic_mode"].value_counts()}')
    print(f'Narrative position distribution:\n{meta["narrative_position"].value_counts()}')
else:
    print('Public subset not found at expected path.')
    print('Generating synthetic demonstration data with correct schema...')
    print('NOTE: Synthetic data does not represent research corpus findings.')

    # Synthetic corpus with realistic feature ranges for demonstration
    np.random.seed(42)
    n_cues = 80

    # Simulate 71-dimensional feature vectors with domain-appropriate scaling
    mfcc_data   = np.random.randn(n_cues, 26) * np.array([15,4]*13)
    chroma_data = np.abs(np.random.randn(n_cues, 27)) * 0.25
    spec_data   = np.abs(np.random.randn(n_cues, 6))  * np.array([800,300,1200,400,2000,600])
    rms_data    = np.abs(np.random.randn(n_cues, 4))  * np.array([0.08,0.04,0.18,0.12])
    zcr_data    = np.abs(np.random.randn(n_cues, 2))  * np.array([0.06,0.03])
    tempo_data  = np.abs(np.random.randn(n_cues, 1))  * 25 + 90
    hp_data     = np.abs(np.random.randn(n_cues, 3))  * np.array([0.4,0.15,0.8])

    X = np.hstack([mfcc_data, chroma_data, spec_data, rms_data, zcr_data, tempo_data, hp_data])

    modes     = np.random.choice(['major','minor','modal','atonal'], n_cues, p=[0.15,0.45,0.28,0.12])
    positions = np.random.choice(['opening','rising','climax','falling','resolution'], n_cues, p=[0.12,0.28,0.22,0.20,0.18])
    tempos    = np.random.choice(['slow','moderate','fast','variable','static'], n_cues)
    functions = np.random.choice(['underscore','stinger','theme_statement','counterpoint','transition'], n_cues, p=[0.55,0.12,0.14,0.09,0.10])
    ids       = [f'CEL-2026-{str(i+1).zfill(4)}' for i in range(n_cues)]

    meta = pd.DataFrame({'clip_id': ids, 'harmonic_mode': modes,
                         'narrative_position': positions, 'tempo_category': tempos,
                         'narrative_function': functions})
    feature_cols = [f'feat_{i:03d}' for i in range(71)]
    df = pd.concat([meta, pd.DataFrame(X, columns=feature_cols)], axis=1)

    print(f'Synthetic demonstration corpus: {X.shape[0]} cues × {X.shape[1]} dimensions')
    print(f'Harmonic mode distribution (synthetic):\n{meta["harmonic_mode"].value_counts()}')

---

## 2. Feature Space Summary Statistics

Corpus-level descriptive statistics across the 71-dimensional feature space.
No emotion labels are involved in this analysis.

In [ ]:
# Per-feature descriptive statistics
feat_df = pd.DataFrame(X, columns=feature_cols)
stats_df = feat_df.describe().T
stats_df['skewness'] = feat_df.apply(stats.skew)
stats_df['kurtosis'] = feat_df.apply(stats.kurtosis)
stats_df['cv'] = stats_df['std'] / stats_df['mean'].abs()  # coefficient of variation

print('Feature space summary (first 10 dimensions):')
print(stats_df[['mean','std','min','max','skewness','kurtosis','cv']].head(10).round(3))

print(f'\nHighest variance features (top 10):')
print(stats_df['std'].nlargest(10))

print(f'\nLowest variance features (bottom 10):')
print(stats_df['std'].nsmallest(10))

---

## 3. Per-Feature-Group Distribution Visualization

Distribution plots for each of the seven acoustic feature groups.
Displayed as violin plots to show both spread and density.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
fig.patch.set_facecolor(CEL_PALETTE['background'])
fig.suptitle('Cinematic Emotion Lab · Feature Group Distributions\nPublic Corpus Subset — Structural Analysis Only',
             color=CEL_PALETTE['text'], fontsize=14, fontweight='bold', y=1.01)

group_items = list(FEATURE_GROUPS.items())
group_colors = [CEL_PALETTE['gold'], CEL_PALETTE['blue'], CEL_PALETTE['red'],
                '#2d5a3d', '#5c3d1a', '#3d1a5c', CEL_PALETTE['muted']]

for idx, ((group_name, col_indices), color) in enumerate(zip(group_items, group_colors)):
    if idx >= 7:
        break
    ax = axes[idx // 3][idx % 3]
    group_data = X[:, col_indices]

    # Normalize for comparable display within group
    parts = ax.violinplot(group_data, showmedians=True, showextrema=True)

    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
        pc.set_edgecolor(CEL_PALETTE['text'])
        pc.set_linewidth(0.5)

    parts['cmedians'].set_color(CEL_PALETTE['gold'])
    parts['cmedians'].set_linewidth(1.5)

    ax.set_title(f'{group_name}  ({len(col_indices)}d)', color=CEL_PALETTE['text'], pad=8)
    ax.set_xlabel('Feature dimension index', fontsize=8)
    ax.set_ylabel('Normalized value', fontsize=8)

# Hide unused subplots
for i in range(len(group_items), 9):
    axes[i // 3][i % 3].set_visible(False)

plt.tight_layout()
plt.savefig('../../visualizations/plots/NB-EDA-001_feature-group-distributions.png',
            dpi=150, bbox_inches='tight', facecolor=CEL_PALETTE['background'])
plt.show()
print('Figure saved: NB-EDA-001_feature-group-distributions.png')

---

## 4. Feature Variance Profile

Variance per dimension across the full 71-dimensional space.
High-variance dimensions carry more discriminative signal.
Low-variance dimensions are candidates for dimensionality reduction.

> **Disclosure note:** This analysis shows feature variance across the corpus,
> not feature discriminability between emotion categories. Discriminability
> analysis is part of EXP-002 and remains embargoed.

In [ ]:
variances = np.var(X, axis=0)

# Build feature group coloring for the variance profile
dim_colors = []
for i in range(X.shape[1]):
    for group_name, indices in FEATURE_GROUPS.items():
        if i in indices:
            dim_colors.append(group_colors[list(FEATURE_GROUPS.keys()).index(group_name)])
            break

fig, ax = plt.subplots(figsize=(18, 5))
fig.patch.set_facecolor(CEL_PALETTE['background'])

bars = ax.bar(range(71), variances, color=dim_colors, alpha=0.8, width=0.85)

ax.set_xlabel('Feature dimension (0-70)', fontsize=10)
ax.set_ylabel('Variance', fontsize=10)
ax.set_title('Cinematic Emotion Lab · Feature Variance Profile (71 dimensions)\n'
             'Color-coded by acoustic domain · Public corpus subset',
             color=CEL_PALETTE['text'], pad=10)

# Group boundary lines
boundaries = [0, 26, 53, 59, 63, 65, 66, 69]
group_names_ordered = list(FEATURE_GROUPS.keys())
for b, name in zip(boundaries[:-1], group_names_ordered):
    ax.axvline(x=b - 0.5, color=CEL_PALETTE['muted'], linewidth=0.8, linestyle='--', alpha=0.5)

# Legend
patches = [mpatches.Patch(color=c, label=n) for n, c in zip(group_names_ordered, group_colors)]
ax.legend(handles=patches, loc='upper right', framealpha=0.3,
          labelcolor=CEL_PALETTE['text'], facecolor=CEL_PALETTE['surface'])

plt.tight_layout()
plt.savefig('../../visualizations/plots/NB-EDA-001_variance-profile.png',
            dpi=150, bbox_inches='tight', facecolor=CEL_PALETTE['background'])
plt.show()
print('Figure saved: NB-EDA-001_variance-profile.png')

---

## 5. Feature Correlation Matrix

Pairwise correlation between feature dimensions. Within-group and between-group
correlation structure informs subsequent dimensionality reduction decisions.

> No emotion labels are used in this correlation analysis.

In [ ]:
corr_matrix = np.corrcoef(X.T)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix,
    colorscale=[
        [0.0,  '#1a3a5c'],
        [0.25, '#0a0a0a'],
        [0.5,  '#111111'],
        [0.75, '#3a1a0a'],
        [1.0,  '#8b1a1a'],
    ],
    zmid=0,
    zmin=-1, zmax=1,
    colorbar=dict(title='Pearson r', tickcolor=CEL_PALETTE['text'],
                  titlefont=dict(color=CEL_PALETTE['text']),
                  tickfont=dict(color=CEL_PALETTE['text']))
))

# Group boundary annotations
group_boundaries = [0, 26, 53, 59, 63, 65, 66, 69, 71]
for b in group_boundaries[1:-1]:
    fig.add_hline(y=b - 0.5, line_color=CEL_PALETTE['gold'], line_width=0.8, opacity=0.5)
    fig.add_vline(x=b - 0.5, line_color=CEL_PALETTE['gold'], line_width=0.8, opacity=0.5)

fig.update_layout(
    title=dict(
        text='Cinematic Emotion Lab · Feature Correlation Matrix (71×71)<br>'
             '<sup>Gold lines mark acoustic domain boundaries · Public corpus subset</sup>',
        font=dict(color=CEL_PALETTE['text'], size=14)
    ),
    paper_bgcolor=CEL_PALETTE['background'],
    plot_bgcolor=CEL_PALETTE['surface'],
    font=dict(color=CEL_PALETTE['text']),
    width=800, height=750,
    xaxis=dict(title='Feature dimension', color=CEL_PALETTE['text']),
    yaxis=dict(title='Feature dimension', color=CEL_PALETTE['text']),
)

fig.write_html('../../visualizations/plots/NB-EDA-001_correlation-matrix.html')
fig.show()
print('Interactive figure saved: NB-EDA-001_correlation-matrix.html')

---

## 6. Feature Distributions by Harmonic Mode

Distribution of selected acoustic features split by `harmonic_mode` annotation
(major / minor / modal / atonal). This is a structural metadata field - not
an emotion label - and is included in the public subset.

This analysis tests a structural hypothesis: if harmonic mode leaves measurable
traces in the acoustic feature space, we should observe systematic differences
in Chroma CQT and MFCC distributions across mode categories.

In [ ]:
# Select representative features: Chroma CQT mean (dim 26), MFCC_0 mean (dim 0),
# Spectral centroid mean (dim 53), RMS mean (dim 59), Tempo (dim 65)
ANALYSIS_FEATURES = {
    'MFCC_0 (mean)':            0,
    'MFCC_1 (mean)':            2,
    'Chroma CQT C (mean)':      26,
    'Chroma CQT G (mean)':      32,
    'Spectral Centroid (mean)': 53,
    'RMS Energy (mean)':        59,
    'Tempo':                    65,
    'H/P Ratio (mean)':         66,
}

modes = meta['harmonic_mode'].values
mode_cats = ['major', 'minor', 'modal', 'atonal']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.patch.set_facecolor(CEL_PALETTE['background'])
fig.suptitle('Cinematic Emotion Lab · Feature Distributions by Harmonic Mode\n'
             'Structural metadata split (not emotion labels) · Public corpus subset',
             color=CEL_PALETTE['text'], fontsize=13)

for idx, (feat_name, feat_idx) in enumerate(ANALYSIS_FEATURES.items()):
    ax = axes[idx // 4][idx % 4]
    feat_values = X[:, feat_idx]

    for mode in mode_cats:
        mask = modes == mode
        if mask.sum() > 1:
            vals = feat_values[mask]
            ax.hist(vals, bins=12, alpha=0.55, color=MODE_COLORS[mode],
                    label=f'{mode} (n={mask.sum()})', density=True, edgecolor='none')

    ax.set_title(feat_name, color=CEL_PALETTE['text'], fontsize=9)
    ax.legend(fontsize=7, framealpha=0.3,
              labelcolor=CEL_PALETTE['text'], facecolor=CEL_PALETTE['surface'])

plt.tight_layout()
plt.savefig('../../visualizations/plots/NB-EDA-001_features-by-harmonic-mode.png',
            dpi=150, bbox_inches='tight', facecolor=CEL_PALETTE['background'])
plt.show()
print('Figure saved: NB-EDA-001_features-by-harmonic-mode.png')

---

## 7. Feature Distributions by Narrative Position

Distribution of selected acoustic features split by `narrative_position`
(opening / rising / climax / falling / resolution).

This analysis is directly relevant to RQ4: *Does narrative position leave
measurable and predictive traces in the acoustic signal?*
The analysis is structural — it examines whether the feature distributions
differ across narrative positions without revealing emotion label data.

In [ ]:
positions = meta['narrative_position'].values
position_cats = ['opening', 'rising', 'climax', 'falling', 'resolution']

# Box plots: RMS energy, spectral centroid, tempo, and H/P ratio by narrative position
POSITION_FEATURES = {
    'RMS Energy (mean)':        59,
    'RMS Dynamic Range':        62,
    'Spectral Centroid (mean)': 53,
    'Tempo':                    65,
    'H/P Ratio (mean)':         66,
    'ZCR (mean)':               63,
}

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=list(POSITION_FEATURES.keys()),
    vertical_spacing=0.18,
    horizontal_spacing=0.08
)

for idx, (feat_name, feat_idx) in enumerate(POSITION_FEATURES.items()):
    row, col = divmod(idx, 3)
    for pos in position_cats:
        mask = positions == pos
        if mask.sum() > 0:
            fig.add_trace(go.Box(
                y=X[mask, feat_idx],
                name=pos,
                marker_color=NARRATIVE_COLORS[pos],
                showlegend=(idx == 0),
                boxmean='sd',
                line_width=1.2,
            ), row=row + 1, col=col + 1)

fig.update_layout(
    title=dict(
        text='Cinematic Emotion Lab · Feature Distributions by Narrative Position<br>'
             '<sup>RQ4 structural analysis · Public corpus subset · No emotion labels</sup>',
        font=dict(color=CEL_PALETTE['text'], size=14)
    ),
    paper_bgcolor=CEL_PALETTE['background'],
    plot_bgcolor=CEL_PALETTE['surface'],
    font=dict(color=CEL_PALETTE['text']),
    legend=dict(
        title='Narrative position',
        font=dict(color=CEL_PALETTE['text']),
        bgcolor=CEL_PALETTE['surface'],
    ),
    height=580, width=1050,
)
fig.update_annotations(font_color=CEL_PALETTE['text'])

fig.write_html('../../visualizations/plots/NB-EDA-001_features-by-narrative-position.html')
fig.show()
print('Interactive figure saved: NB-EDA-001_features-by-narrative-position.html')

---

## 8. MFCC Mean Profile Across the Corpus

The 13 MFCC coefficient means, averaged across the full public subset.
Individual-cue MFCC profiles will be visualized in NB-VIZ-002 (target: v0.15).

In [ ]:
# MFCC mean coefficients (dims 0, 2, 4, ... 24 — every other dim is mean)
mfcc_mean_dims = list(range(0, 26, 2))  # 13 mean coefficients
mfcc_std_dims  = list(range(1, 26, 2))  # 13 std coefficients

corpus_mfcc_mean = X[:, mfcc_mean_dims].mean(axis=0)
corpus_mfcc_std  = X[:, mfcc_mean_dims].std(axis=0)

fig = go.Figure()
coeff_indices = list(range(1, 14))

fig.add_trace(go.Scatter(
    x=coeff_indices,
    y=corpus_mfcc_mean,
    mode='lines+markers',
    name='Corpus mean',
    line=dict(color=CEL_PALETTE['gold'], width=2.5),
    marker=dict(size=7, color=CEL_PALETTE['gold']),
))

fig.add_trace(go.Scatter(
    x=coeff_indices + coeff_indices[::-1],
    y=list(corpus_mfcc_mean + corpus_mfcc_std) + list((corpus_mfcc_mean - corpus_mfcc_std)[::-1]),
    fill='toself',
    fillcolor='rgba(201,162,39,0.12)',
    line=dict(color='rgba(0,0,0,0)'),
    name='±1 std',
    showlegend=True,
))

fig.update_layout(
    title=dict(
        text='Cinematic Emotion Lab · MFCC Mean Coefficient Profile<br>'
             '<sup>Corpus average with ±1 std envelope · 13 coefficients · Public subset</sup>',
        font=dict(color=CEL_PALETTE['text'], size=13)
    ),
    paper_bgcolor=CEL_PALETTE['background'],
    plot_bgcolor=CEL_PALETTE['surface'],
    font=dict(color=CEL_PALETTE['text']),
    xaxis=dict(title='MFCC coefficient index', dtick=1, color=CEL_PALETTE['text'],
               gridcolor='#1a1a1a'),
    yaxis=dict(title='Mean coefficient value', color=CEL_PALETTE['text'],
               gridcolor='#1a1a1a'),
    legend=dict(font=dict(color=CEL_PALETTE['text']), bgcolor=CEL_PALETTE['surface']),
    height=400, width=750,
)

fig.write_html('../../visualizations/plots/NB-EDA-001_mfcc-mean-profile.html')
fig.show()
print('Interactive figure saved: NB-EDA-001_mfcc-mean-profile.html')

---

## 9. Chroma CQT Energy Distribution

Mean chroma energy across the 12 pitch classes, visualized as a polar radar chart.
Shows the aggregate harmonic center of gravity of the corpus.
The full per-cue chroma radar system is implemented in NB-VIZ-002.

In [ ]:
PITCH_CLASSES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']

# Chroma CQT mean values: dims 26-37 (12 pitch class means)
chroma_mean_dims = list(range(26, 38))
corpus_chroma = X[:, chroma_mean_dims].mean(axis=0)
corpus_chroma_norm = corpus_chroma / corpus_chroma.max()

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=list(corpus_chroma_norm) + [corpus_chroma_norm[0]],
    theta=PITCH_CLASSES + [PITCH_CLASSES[0]],
    fill='toself',
    fillcolor='rgba(201,162,39,0.18)',
    line=dict(color=CEL_PALETTE['gold'], width=2),
    name='Corpus mean chroma'
))

fig.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 1.05], color=CEL_PALETTE['muted'],
                        gridcolor='#1a1a1a', tickfont=dict(color=CEL_PALETTE['muted'])),
        angularaxis=dict(color=CEL_PALETTE['text'], gridcolor='#1a1a1a',
                         tickfont=dict(color=CEL_PALETTE['text'])),
        bgcolor=CEL_PALETTE['surface'],
    ),
    paper_bgcolor=CEL_PALETTE['background'],
    title=dict(
        text='Cinematic Emotion Lab · Corpus Chroma CQT Profile<br>'
             '<sup>Mean normalized energy per pitch class · Public subset</sup>',
        font=dict(color=CEL_PALETTE['text'], size=13)
    ),
    font=dict(color=CEL_PALETTE['text']),
    legend=dict(font=dict(color=CEL_PALETTE['text']), bgcolor=CEL_PALETTE['surface']),
    height=480, width=480,
)

fig.write_html('../../visualizations/plots/NB-EDA-001_chroma-radar.html')
fig.show()
print('Interactive figure saved: NB-EDA-001_chroma-radar.html')

---

## 10. Corpus Statistics Summary

Aggregate statistics suitable for public disclosure.
No per-clip labels are included.

In [ ]:
print('=' * 60)
print('CINEMATIC EMOTION LAB - PUBLIC CORPUS SUBSET STATISTICS')
print('v0.2 Release · August 2026')
print('=' * 60)

print(f'\nCorpus size (public subset): {X.shape[0]} cinematic audio cues')
print(f'Feature dimensions:          {X.shape[1]}')
print(f'Feature groups:              7 acoustic domains')

print('\n--- Harmonic Mode Distribution ---')
for mode, count in meta['harmonic_mode'].value_counts().items():
    pct = count / len(meta) * 100
    bar = '|' * int(pct / 2)
    print(f'  {mode:<8} {bar:<25} {count:3d} ({pct:.1f}%)')

print('\n--- Narrative Position Distribution ---')
for pos, count in meta['narrative_position'].value_counts().items():
    pct = count / len(meta) * 100
    bar = '|' * int(pct / 2)
    print(f'  {pos:<12} {bar:<20} {count:3d} ({pct:.1f}%)')

print('\n--- Narrative Function Distribution ---')
for func, count in meta['narrative_function'].value_counts().items():
    pct = count / len(meta) * 100
    bar = '|' * int(pct / 2)
    print(f'  {func:<22} {bar:<15} {count:3d} ({pct:.1f}%)')

print('\n--- Feature Space Summary ---')
print(f'  Global mean:         {X.mean():.4f}')
print(f'  Global std:          {X.std():.4f}')
print(f'  Dimension with highest variance: {np.argmax(np.var(X, axis=0))}')
print(f'  Dimensions with near-zero variance (std < 0.001): {(np.std(X, axis=0) < 0.001).sum()}')

print('\n--- Withheld ---')
print('  primary_emotion:         withheld pending publication')
print('  emotional_intensity:     withheld pending publication')
print('  orchestration_density:   withheld pending publication')
print('  full corpus size:        withheld')
print('  classification results:  withheld pending publication')

print('\n' + '=' * 60)

---

## 11. Observations and Forward Direction

> The following section contains qualitative structural observations from this
> exploratory analysis. It does not contain classification results, performance
> metrics, or findings from EXP-002, EXP-003, or EXP-004.

### Structural Observations (Public)

**Feature variance is non-uniform across dimensions.**
The MFCC and Chroma CQT groups exhibit the highest within-group variance
in the public subset, consistent with their role as the primary carriers of
timbral and harmonic information in the feature space. Spectral and RMS
dimensions show compressed variance ranges.

**Inter-group correlation is present but bounded.**
The correlation matrix reveals expected structural relationships: MFCC dimensions
show sequential correlation consistent with their DCT basis. Chroma dimensions
show neighbor correlations consistent with tonal proximity. Cross-group
correlations are present at lower magnitudes, suggesting that the seven acoustic
domains capture partially but not wholly redundant information.

**Narrative position shows distributional variation in dynamic features.**
Preliminary visual inspection suggests that RMS energy and dynamic range
distributions vary across narrative positions in the public subset. This is
consistent with the hypothesis underlying RQ4. Formal statistical testing
is deferred to EXP-002.

**Harmonic mode is partially recoverable from Chroma CQT features.**
The Chroma CQT distributions by harmonic mode show visible separation in
certain pitch classes, particularly for the atonal category. This is expected
given the theoretical relationship between mode and pitch-class energy profiles.

### Forward Direction

This exploratory analysis informs three downstream decisions:

1. **Feature scaling strategy for EXP-002:** The non-uniform variance across
   groups suggests that per-group standardization may be preferable to global
   standardization before classification.

2. **Dimensionality reduction candidate identification:** Dimensions with
   near-zero corpus variance are candidates for removal before modeling.
   Final selection will be made with reference to per-class variance, not
   corpus-level variance alone.

3. **Narrative position as a conditioning variable:** The distributional
   evidence from Section 7 supports including narrative position as a
   conditioning variable in EXP-002, rather than treating it as an
   independent predictor.

---

*NB-EDA-001 · Cinematic Emotion Lab · v0.2 Release · August 2026*  
*Bernard G. - Film Composer · AI Architect · Computational Musicology Researcher*